In [1]:
import pandas as pd
from hypernetx import Hypergraph

def build_H_from_edge_sets(E):
    """
    PAPER: Build hypergraph H=(V,𝓔) from a hyperedge family 𝓔 given as:
        E = { edge_id : set_of_vertices }.
    Implementation uses an incidence table of pairs (e,v) with v ∈ e.
    """
    rows = []
    for e_id, vertices in E.items():
        for v in vertices:
            rows.append({
                "edges": e_id,
                "nodes": v,
                "weight": 1,
                "misc_properties": None
            })
    return Hypergraph(pd.DataFrame(rows))

def edge_sets(H):
    """Return the family 𝓔(H) as a dict: e -> set of vertices in e."""
    return {e: set(verts) for e, verts in H.incidence_dict.items()}

def print_hyperedges(H, title="Hyperedges 𝓔(H)"):
    """PAPER: print each hyperedge e ∈ 𝓔(H) as a subset of V(H)."""
    print(title)
    for e, verts in edge_sets(H).items():
        print(f"  {e}: {sorted(verts)}")

Lets create a Hypergraph with contained or wholly contained hyperedges

In [3]:
E = {
    "e1": {"a","b","c","d"},
    "e2": {"a","b"},        # e2 ⊂ e1
    "e3": {"b","c"},        # e3 ⊂ e1
    "e4": {"c","d","e"},
    "e5": {"d","e"},        # e5 ⊂ e4
    "e6": {"f"},            # singleton (optional to keep/remove later)
}

H = build_H_from_edge_sets(E)
print_hyperedges(H, "Original hypergraph H:")

Original hypergraph H:
  e1: ['a', 'b', 'c', 'd']
  e2: ['a', 'b']
  e3: ['b', 'c']
  e4: ['c', 'd', 'e']
  e5: ['d', 'e']
  e6: ['f']


Lets attempt Minimization

In [5]:
def minimize_by_inclusion(H, keep_equal=False):
    """
    PAPER: Construct M(H) by removing contained hyperedges.
    We keep a hyperedge e if there is NO hyperedge f such that e ⊂ f.
    
    Parameters:
      keep_equal = False:
        If two different edge IDs have exactly the same vertex set,
        we treat them as duplicates and keep only one representative.
      keep_equal = True:
        Keep all equal edges (rarely needed in our setting).
    """
    E = edge_sets(H)  # e -> set of vertices
    
    keep = []
    edge_ids = list(E.keys()) # list of hyperedges
    
    for e in edge_ids:
        Se = E[e] # set of vertices in hyperedge e
        contained = False
        
        for f in edge_ids:
            if e == f: # check if we are comparing the same edge, if so skip to avoid self-comparison
                continue
            Sf = E[f] # set of vertices in hyperedge f
            
            # check if Se is a subset of Sf, if so e is contained in f and we can stop checking further for this edge e
            if Se < Sf:
                contained = True
                break
            
            # else if the sets are equal and we are not keeping equal edges,
            # we use string comparison of edge IDs to deterministically keep only one of the duplicates
            if (not keep_equal) and (Se == Sf) and (str(e) > str(f)):
                contained = True
                break
        
        # if after checking all other edges f, we find that e is not contained in any of them, we add e to the list of edges to keep
        if not contained:
            keep.append(e)
    
    return H.restrict_to_edges(keep)

H_hat_manual = minimize_by_inclusion(H)
print_hyperedges(H, "\nOriginal hypergraph H:")
print_hyperedges(H_hat_manual, "\nMinimized M(H) (manual subset test):")


Original hypergraph H:
  e1: ['a', 'b', 'c', 'd']
  e2: ['a', 'b']
  e3: ['b', 'c']
  e4: ['c', 'd', 'e']
  e5: ['d', 'e']
  e6: ['f']

Minimized M(H) (manual subset test):
  e1: ['a', 'b', 'c', 'd']
  e4: ['c', 'd', 'e']
  e6: ['f']


Lets Attempt Minimization using built-in method from hypernetx - .toplexes()

In [8]:
def minimize_with_toplexes(H):
    """
    HyperNetX: toplexes() returns inclusion-maximal hyperedges (facets).
    PAPER: These are exactly the hyperedges not contained in any other.
    """
    maximal_edges = H.toplexes()
    return H.restrict_to_edges(maximal_edges)

H_hat = minimize_with_toplexes(H)
print_hyperedges(H, "\nOriginal hypergraph H:")
print_hyperedges(H_hat, "\nMinimized M'(H) (toplexes):")


Original hypergraph H:
  e1: ['a', 'b', 'c', 'd']
  e2: ['a', 'b']
  e3: ['b', 'c']
  e4: ['c', 'd', 'e']
  e5: ['d', 'e']
  e6: ['f']

Minimized M'(H) (toplexes):
  e1: ['a', 'b', 'c', 'd']
  e4: ['c', 'd', 'e']
  e6: ['f']


Quick Comparison

In [10]:
# create sets of edges in M(H) and M'(H) to compare if they are the same
manual_edges = set(edge_sets(H_hat_manual).keys())
toplex_edges  = set(edge_sets(H_hat).keys())

print("Manual kept edges :", manual_edges)
print("Toplex kept edges  :", toplex_edges)
print("Agree?", manual_edges == toplex_edges)

Manual kept edges : {'e1', 'e6', 'e4'}
Toplex kept edges  : {'e1', 'e6', 'e4'}
Agree? True


Lets createe the duals

In [11]:
H_star = H.dual()

# Strategy 1: minimize first, then dual
S1 = minimize_with_toplexes(H).dual()

# Strategy 2: dual first, then minimize
S2 = minimize_with_toplexes(H_star)

print_hyperedges(H_star, "\nDual H* (not minimized):")
print_hyperedges(S1, "\nStrategy 1: (minimize H) then dual")
print_hyperedges(S2, "\nStrategy 2: dual then (minimize H*)")


Dual H* (not minimized):
  a: ['e1', 'e2']
  b: ['e1', 'e2', 'e3']
  c: ['e1', 'e3', 'e4']
  d: ['e1', 'e4', 'e5']
  e: ['e4', 'e5']
  f: ['e6']

Strategy 1: (minimize H) then dual
  a: ['e1']
  b: ['e1']
  c: ['e1', 'e4']
  d: ['e1', 'e4']
  e: ['e4']
  f: ['e6']

Strategy 2: dual then (minimize H*)
  b: ['e1', 'e2', 'e3']
  c: ['e1', 'e3', 'e4']
  d: ['e1', 'e4', 'e5']
  f: ['e6']


We still need to remove singletons

In [ ]:
def remove_singleton_edges(H):
    """
    PAPER: Remove hyperedges e with |e|=1.
    (Use only if your model assumes edges have size >= 2.)
    """
    E = edge_sets(H)
    keep = [e for e, verts in E.items() if len(verts) >= 2]
    return H.restrict_to_edges(keep)

H_hat_no_singletons = remove_singleton_edges(H_hat)
print_hyperedges(H_hat_no_singletons, "\nMinimized M(H) with singleton edges removed:")


Minimized Ĥ with singleton edges removed:
  e1: ['a', 'b', 'c', 'd']
  e4: ['c', 'd', 'e']


In [15]:
H_star_no_singletons = remove_singleton_edges(H_star)
print_hyperedges(H_star_no_singletons, "\nDual H* with singleton edges removed:") 

S1_no_singletons = remove_singleton_edges(S1)
print_hyperedges(S1, "\nStrategy 1_1: (minimize H) then dual, with singleton edges not removed:")
print_hyperedges(S1_no_singletons, "\nStrategy 1_2: (minimize H) then dual, with singleton edges removed:")

S2_no_singletons = remove_singleton_edges(S2)
print_hyperedges(S2, "\nStrategy 2_1: dual then (minimize H*), with singleton edges not removed:")
print_hyperedges(S2_no_singletons, "\nStrategy 2_2: dual then (minimize H*), with singleton edges removed:")  


Dual H* with singleton edges removed:
  a: ['e1', 'e2']
  b: ['e1', 'e2', 'e3']
  c: ['e1', 'e3', 'e4']
  d: ['e1', 'e4', 'e5']
  e: ['e4', 'e5']

Strategy 1_1: (minimize H) then dual, with singleton edges not removed:
  a: ['e1']
  b: ['e1']
  c: ['e1', 'e4']
  d: ['e1', 'e4']
  e: ['e4']
  f: ['e6']

Strategy 1_2: (minimize H) then dual, with singleton edges removed:
  c: ['e1', 'e4']
  d: ['e1', 'e4']

Strategy 2_1: dual then (minimize H*), with singleton edges not removed:
  b: ['e1', 'e2', 'e3']
  c: ['e1', 'e3', 'e4']
  d: ['e1', 'e4', 'e5']
  f: ['e6']

Strategy 2_2: dual then (minimize H*), with singleton edges removed:
  b: ['e1', 'e2', 'e3']
  c: ['e1', 'e3', 'e4']
  d: ['e1', 'e4', 'e5']


From what I can see, Strategy 2 is the result we need.
It is important to note that this particular H is not Helly due to vertex f being in its own bubble.

Assuming f didnt exist, the edges obtained from Strategy 2_2 correctly represents the hyperegdes suitable for reduction.

In [20]:
# Example usage:
# PAPER PIPELINE:
# 1) Start with graph G (HCH chordal), compute maximal cliques 𝓚(G).
# 2) Construct hypergraph H where V(H)=𝓚(G) (cliques become vertices of H).
# 3) Dualize: H*.
# 4) Minimize inside the dual: Ĥ*.

# Toy H: vertices are "cliques" K1,K2,K3,... (labels stand in for maximal cliques)
E_H = {
    # hyperedges of H (these typically correspond to vertices of G,
    # each listing the maximal cliques that contain that vertex)
    "v1": {"K1", "K2", "K3"},
    "v2": {"K2", "K3"},
    "v3": {"K3", "K4"},
    "v4": {"K1", "K3"},
}

H = build_H_from_edge_sets(E_H)
print_hyperedges(H, "H (toy): vertices are maximal cliques of G")

H_star = H.dual()
print_hyperedges(H_star, "\nH* (dual): this is where we run leaf logic")

H_star_hat = minimize_with_toplexes(H_star)
print_hyperedges(H_star_hat, "\nM(H*) = minimized dual (Strategy 2)")

H (toy): vertices are maximal cliques of G
  v1: ['K1', 'K2', 'K3']
  v2: ['K2', 'K3']
  v3: ['K3', 'K4']
  v4: ['K1', 'K3']

H* (dual): this is where we run leaf logic
  K1: ['v1', 'v4']
  K2: ['v1', 'v2']
  K3: ['v1', 'v2', 'v3', 'v4']
  K4: ['v3']

M(H*) = minimized dual (Strategy 2)
  K3: ['v1', 'v2', 'v3', 'v4']
